# AgriNav — Baseline Control: is the custom architecture the bottleneck?

The `ImageNet→RiceSEG` pretraining reached **mIoU 0.5816**, but **weeds IoU 0.34** — and weeds swung between **0.11 and 0.34** on consecutive epochs while loss fell smoothly. Two explanations fit:

| | Explanation | What it implies |
|---|---|---|
| **(a)** | The task is **data-limited** — RiceSEG lacks weed pixels, so any model plateaus | Go collect/annotate weeds. The architecture is fine. |
| **(b)** | The **custom Det-ResNet-50 is the limit** — a stock pretrained segmenter does better | Change the model *before* building a detector on it. |

This notebook runs the experiment that discriminates them, exactly as the project's own v5 optimization report proposed: *"If a pretrained baseline strongly outperforms WeedDet, the custom architecture is the bottleneck."*

**Everything is held fixed except the model.** `training/baseline_seg_control.py` *imports* the split, class weights, dataset, loss, and metric from `riceseg_pretrain.py` rather than reimplementing them — a reimplementation would silently invalidate the comparison. Same seed 42, same group-aware split, same CE+Dice loss, same AdamW+cosine, same `ConfMat.iou()`.

**This produces a measurement, not a backbone.** No WeedDet backbone is exported, by design.

**Before you run:** `Runtime → Change runtime type → GPU`, and `RiceSEG.zip` in `MyDrive/agrinav_data/`.

## 1. Confirm GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the code

Stops immediately if the clone fails, so later cells cannot cascade into confusing errors. If the repo is private, add a `GITHUB_TOKEN` Colab secret (🔑 in the sidebar) holding a fine-grained PAT with **Contents: Read**; the token is never printed and is stripped from the git remote after cloning.

**This requires `training/baseline_seg_control.py` to be pushed to the branch.** The assert below fails loudly if it is not.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/Bmerrysmith/Autonomous-tractor-system.git'
BRANCH   = 'codex/repository-recovery'
REPO_DIR = '/content/agrinav'

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
print('GitHub token found in Colab secrets:', bool(token))

def _redact(s):
    return s.replace(token, '***') if token else s

env = dict(os.environ, GIT_TERMINAL_PROMPT='0')   # auth failure -> error, not hang
auth_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '--branch', BRANCH, auth_url, REPO_DIR],
                       env=env, capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit('CLONE FAILED — stopping.\n\n' + _redact(r.stderr))
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL], env=env)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], env=env)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], env=env)
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], env=env)

os.chdir(REPO_DIR)
assert os.path.exists('training/baseline_seg_control.py'), (
    'baseline_seg_control.py is not on this branch. It must be committed and pushed '
    'before this notebook can run.')
print('repo ready at', REPO_DIR)
!git log --oneline -1

## 4. Dependencies

`transformers` is only needed for the optional SegFormer arm.

In [ ]:
!pip install -q numpy Pillow tqdm transformers

## 5. Extract RiceSEG

Same archive, same SHA-256 gate as the pretraining run — a different archive would invalidate the comparison. Extracts to fast local disk, not Drive.

In [ ]:
import hashlib, zipfile, glob, os

RICESEG_ZIP  = '/content/drive/MyDrive/agrinav_data/RiceSEG.zip'
EXPECTED_SHA = '0071d9f941508afd9a86aa5ef740433dee938e1c7a9508a191d3e46a2909be96'
DATA_ROOT    = '/content/RiceSEG'

if not os.path.exists(RICESEG_ZIP):
    hits = glob.glob('/content/drive/MyDrive/**/RiceSEG.zip', recursive=True)
    assert hits, 'RiceSEG.zip not found in Drive.'
    RICESEG_ZIP = hits[0]
print('using', RICESEG_ZIP)

if not os.path.isdir(DATA_ROOT):
    h = hashlib.sha256()
    with open(RICESEG_ZIP, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    print('sha256:', h.hexdigest())
    assert h.hexdigest() == EXPECTED_SHA, 'hash mismatch — this is not the archive the pretraining used.'
    os.makedirs(DATA_ROOT, exist_ok=True)
    with zipfile.ZipFile(RICESEG_ZIP) as z:
        z.extractall(DATA_ROOT)
print('data at', DATA_ROOT)
print(os.listdir(os.path.join(DATA_ROOT, 'global rice segmentation')))

## 6. Smoke test (2 epochs)

Catches OOM, shape bugs, and weight-download failures in a few minutes instead of discovering them two hours into the real run. **If this fails, stop and fix — do not run cell 7.**

Lower `--batch-size` to 4 if you hit CUDA OOM.

In [ ]:
# -u keeps output streaming. Without it Colab block-buffers and a healthy run
# looks identical to a hang.
!python -B -u training/baseline_seg_control.py \
  --data-root "{DATA_ROOT}" \
  --arch deeplabv3_resnet50 \
  --epochs 2 --batch-size 8 --img-size 512 \
  --out /content/smoke_deeplabv3.json

## 7. The control run — DeepLabV3-ResNet50, 30 epochs

Matches the pretraining run's epochs / img-size / lr / val-ratio / seed.

**Expect roughly 2–4 hours on a T4.** DeepLabV3-R50 is heavier than the custom network, so this may run longer than the original 30-epoch run did. Keep the tab open; Colab drops idle runtimes.

In [ ]:
OUT_DIR = '/content/drive/MyDrive/agrinav_data/out'
os.makedirs(OUT_DIR, exist_ok=True)

!python -B -u training/baseline_seg_control.py \
  --data-root "{DATA_ROOT}" \
  --arch deeplabv3_resnet50 \
  --epochs 30 --batch-size 8 --img-size 512 --lr 3e-4 --val-ratio 0.1 --seed 42 \
  --stability-window 5 \
  --out "{OUT_DIR}/baseline_deeplabv3_resnet50.json"

## 8. (Optional) Second arm — SegFormer-B2

A transformer pretrained on ADE20K. Run this only if you want a second, architecturally different data point — it roughly doubles total GPU time. Two agreeing baselines are much stronger evidence than one.

Pin `--hf-revision` to a commit hash if you intend to cite the result.

In [ ]:
!python -B -u training/baseline_seg_control.py \
  --data-root "{DATA_ROOT}" \
  --arch segformer_b2 \
  --epochs 30 --batch-size 8 --img-size 512 --lr 3e-4 --val-ratio 0.1 --seed 42 \
  --stability-window 5 \
  --out "{OUT_DIR}/baseline_segformer_b2.json"

## 9. The verdict

Compares each baseline against the custom `ImageNet→RiceSEG` run, using the **same stability window** recomputed from the pretraining manifest's own history — so peaks are compared to peaks and stable means to stable means.

Read the **weeds** row, not overall mIoU. Overall mIoU is dominated by `background` and `green_veg`, which every model scores ~0.87 on; they cannot discriminate the hypotheses.

In [ ]:
import json, sys, glob
sys.path.insert(0, 'training')
from baseline_seg_control import stability_stats

WINDOW = 5
CUSTOM = f'{OUT_DIR}/riceseg_backbone.pth.manifest.json'

runs = []
if os.path.exists(CUSTOM):
    m = json.load(open(CUSTOM))
    runs.append(('custom Det-ResNet-50 (ImageNet)', m['best'], stability_stats(m['history'], WINDOW)))
else:
    print('WARNING: pretraining manifest not found at', CUSTOM)

for p in sorted(glob.glob(f'{OUT_DIR}/baseline_*.json')):
    m = json.load(open(p))
    label = m['model']['model_id'].split('/')[-1]
    runs.append((label, m['best'], m.get('stability') or stability_stats(m['history'], WINDOW)))

if not runs:
    raise SystemExit('no runs found — run cells 7/8 first.')

CLASSES = ['background', 'green_veg', 'senescent', 'panicle', 'weeds', 'duckweed']
w0 = max(len(n) for n, _, _ in runs) + 2

print('BEST-EPOCH'.ljust(w0), 'mIoU   ', '  '.join(c[:9].rjust(9) for c in CLASSES))
for name, best, _ in runs:
    pc = best['per_class_iou']
    print(name.ljust(w0), f"{best['miou_present_classes']:.4f} ",
          '  '.join((f"{pc[c]:.3f}" if pc.get(c) is not None else '  n/a').rjust(9) for c in CLASSES))

print()
print(f'STABLE (last {WINDOW} epochs, mean +/- std)')
for name, _, st in runs:
    if not st:
        continue
    print(f"  {name}")
    print(f"     mIoU  {st['miou_mean']:.4f} +/- {st['miou_std']:.4f}")
    for c in ('weeds', 'duckweed', 'senescent'):
        s = st['per_class'][c]
        if s['mean'] is None:
            print(f"     {c:9s} n/a"); continue
        print(f"     {c:9s} {s['mean']:.4f} +/- {s['std']:.4f}   "
              f"(min {s['min']:.3f}, max {s['max']:.3f})")

# ---- interpretation ----
if len(runs) >= 2:
    base = runs[0][2]['per_class']['weeds']['mean']
    print('\n' + '=' * 62)
    for name, _, st in runs[1:]:
        b = st['per_class']['weeds']['mean']
        if base is None or b is None:
            print(f'{name}: weeds not measurable in the window'); continue
        delta = b - base
        pooled = (runs[0][2]['per_class']['weeds']['std'] + st['per_class']['weeds']['std']) / 2
        print(f'{name}: stable weeds IoU {b:.4f} vs custom {base:.4f}  (delta {delta:+.4f})')
        if pooled > 0 and abs(delta) < pooled:
            print('   -> WITHIN the run-to-run noise. No evidence the architecture is the limit.')
            print('      Read as support for (a) DATA-LIMITED: invest in weed annotation.')
        elif delta > 0:
            print('   -> Baseline is BETTER by more than the noise band.')
            print('      Read as support for (b) ARCHITECTURE-LIMITED: reconsider Det-ResNet-50')
            print('      before building the detector on top of it.')
        else:
            print('   -> Custom model is better by more than the noise band.')
            print('      The custom architecture is earning its keep; weeds are data-limited.')
    print('=' * 62)
    print('\nCaveat: one seed each. A delta near the noise band is suggestive, not conclusive —')
    print('rerun with --seed 1 / --seed 2 before making an irreversible architectural decision.')

## What to do with the result

- **Baselines ≈ custom on weeds** → the task is data-limited. Stop tuning architecture; the weed bottleneck is real and the fix is annotation (see `docs/EXTERNAL_DATASETS_AUDIT.md` — the Rice Field weed BD V3 species set and a properly LFS-cloned `WeedDataset` with `Barnyard_Grass` are the leads).
- **Baselines clearly beat custom on weeds** → decide the architecture question *before* detector training. Note a ViT swap is not drop-in: the detector consumes `C3/C4/C5` from `Det-ResNet-50`, so SegFormer/DINOv2 would need a new neck.
- **Either way**, record the manifests in `docs/research/RICESEG_PRETRAIN_RESULTS.md`. A negative result here is a real, publishable finding — it is the evidence that justifies spending on annotation.

The control manifests live in `MyDrive/agrinav_data/out/baseline_*.json` and carry git commit, environment, model id/revision, full per-epoch history, and the stability block.